[![pythonista](img/pythonista.png)](https://www.pythonista.io)

# Tablas, vistas y carga de datos.

## Configuración.

Ejecuta la siguiente celda antes que cualquier otra. Abre en modo de escritura la base de trabajo ```data/curso.duckdb``` y adjunta los datos del curso en sólo lectura.

In [ ]:
import duckdb

%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.displaycon = False

conexion = duckdb.connect('data/curso.duckdb')
conexion.execute("ATTACH IF NOT EXISTS 'data/thelook.duckdb' AS thelook (READ_ONLY)")
for tabla in ['distribution_centers', 'products', 'users', 'orders', 'order_items']:
    conexion.execute(f"CREATE OR REPLACE VIEW {tabla} AS SELECT * FROM thelook.main.{tabla}")
%sql conexion

Hasta ahora sólo se han consultado datos. En ingeniería de datos, el resultado de una transformación se **guarda** para que otros lo consuman. Para ello se usan dos sublenguajes de *SQL*:

* ***DDL*** (*Data Definition Language*): crea, modifica y elimina objetos (```CREATE```, ```ALTER```, ```DROP```).
* ***DML*** (*Data Manipulation Language*): inserta, modifica y elimina filas (```INSERT```, ```UPDATE```, ```DELETE```, ```MERGE```).

## Configuración de esta lección.

En esta lección y la siguiente se crean objetos, por lo que la celda de configuración es distinta:

* Se abre en modo de escritura una base **de trabajo**, ```data/curso.duckdb```.
* Los datos del curso se adjuntan (```ATTACH```) en **sólo lectura**, para no modificarlos por accidente.
* Se crean en la base de trabajo **vistas** con los mismos nombres de las tablas originales (```orders```, ```users```, etc.), de modo que las consultas se escriben igual que en las lecciones anteriores. Las vistas se estudian más adelante en esta lección.

**Importante:** un archivo de *DuckDB* sólo puede abrirse en modo de escritura desde un proceso a la vez. Si aparece un error de bloqueo (*Could not set lock on file*), cierra el *kernel* de los otros notebooks que usen ```curso.duckdb``` (lecciones 7 y 8, y el de soluciones).

## Esquemas.

En el estándar, las tablas y vistas se agrupan en **esquemas**. El nombre completo de una tabla tiene hasta tres niveles:

```
catálogo.esquema.tabla        → curso.analitica.ventas_mensuales
```

La siguiente celda elimina el esquema ```analitica``` si existe, con todo su contenido (```CASCADE```), y lo crea de nuevo. Así la lección puede ejecutarse desde el principio cuantas veces sea necesario.

In [ ]:
%%sql
DROP SCHEMA IF EXISTS analitica CASCADE;
CREATE SCHEMA analitica;

> **Nota de dialecto:** ```IF EXISTS``` no es estándar, pero lo admiten casi todos los motores. En *BigQuery*, un esquema se llama *dataset* y el catálogo es el proyecto de *Google Cloud*.

## Creación de tablas.

### Con un esquema explícito.

Se definen las columnas, sus tipos y sus restricciones.

In [ ]:
%%sql
CREATE TABLE analitica.metas_mensuales (
  anio INTEGER NOT NULL,
  mes INTEGER NOT NULL CHECK (mes BETWEEN 1 AND 12),
  categoria VARCHAR(50) NOT NULL,
  meta_ventas DECIMAL(12, 2),
  PRIMARY KEY (anio, mes, categoria)
);

> **Nota de dialecto:** *BigQuery* usa ```STRING``` en lugar de ```VARCHAR(n)``` y no aplica las restricciones ```PRIMARY KEY``` ni ```CHECK```; *DuckDB* y *PostgreSQL* sí las validan.

### A partir de una consulta (```CREATE TABLE ... AS```).

Es la forma más común de materializar una transformación (*ELT*): la tabla toma su estructura del resultado de la consulta.

In [ ]:
%%sql
CREATE TABLE analitica.ventas_mensuales AS
SELECT EXTRACT(YEAR FROM oi.created_at) AS anio,
  EXTRACT(MONTH FROM oi.created_at) AS mes,
  p.category AS categoria,
  COUNT(*) AS articulos,
  SUM(oi.sale_price) AS ventas
FROM order_items AS oi
JOIN products AS p ON oi.product_id = p.id
WHERE oi.status = 'Complete'
GROUP BY EXTRACT(YEAR FROM oi.created_at), EXTRACT(MONTH FROM oi.created_at), p.category;

In [ ]:
%%sql
SELECT anio, mes, categoria, ROUND(ventas, 2) AS ventas
FROM analitica.ventas_mensuales
ORDER BY anio DESC, mes DESC, ventas DESC
FETCH FIRST 20 ROWS ONLY;

> **Nota de dialecto:** el estándar escribe ```CREATE TABLE t AS (consulta) WITH DATA```; casi todos los motores aceptan la forma corta. Para reemplazar una tabla existente, *DuckDB*, *BigQuery* y *Snowflake* ofrecen ```CREATE OR REPLACE TABLE```, que no es estándar; la alternativa portable es ```DROP TABLE``` seguido de ```CREATE TABLE```.

## Vistas.

Una **vista** es una consulta guardada con nombre. No almacena datos: cada vez que se consulta, se ejecuta la consulta original sobre los datos actuales.

In [ ]:
%%sql
CREATE VIEW analitica.v_clientes_por_pais AS
SELECT country AS pais,
  COUNT(*) AS clientes,
  ROUND(AVG(age), 1) AS edad_promedio
FROM users
GROUP BY country;

In [ ]:
%%sql
SELECT pais, clientes, edad_promedio
FROM analitica.v_clientes_por_pais
ORDER BY clientes DESC;

| | Tabla | Vista |
|---|---|---|
| Almacena datos | Sí | No |
| Datos actualizados | Sólo cuando se recarga | Siempre |
| Costo de consulta | Lee el resultado ya calculado | Ejecuta la consulta original cada vez |
| Uso típico | Resultados costosos de calcular y consultados con frecuencia | Lógica reutilizable; exponer sólo ciertas columnas o filas |

## Manipulación de datos (*DML*).

### ```INSERT```.

In [ ]:
%%sql
INSERT INTO analitica.metas_mensuales (anio, mes, categoria, meta_ventas)
VALUES (2026, 1, 'Jeans', 5000),
       (2026, 1, 'Sweaters', 3000),
       (2026, 2, 'Jeans', 5500);

* Las restricciones protegen la calidad de los datos. Esta inserción falla porque el mes 13 no cumple el ```CHECK```:

```sql
INSERT INTO analitica.metas_mensuales (anio, mes, categoria, meta_ventas)
VALUES (2026, 13, 'Jeans', 5000);

-- Constraint Error: CHECK constraint failed on table metas_mensuales
```

* ```INSERT``` también acepta una consulta como origen. La siguiente fija como meta de diciembre de 2025 las ventas de noviembre de 2025 más un 10 %:

In [ ]:
%%sql
INSERT INTO analitica.metas_mensuales (anio, mes, categoria, meta_ventas)
SELECT anio, mes + 1, categoria, ROUND(ventas * 1.1, 0)
FROM analitica.ventas_mensuales
WHERE anio = 2025 AND mes = 11;

In [ ]:
%%sql
SELECT anio, mes, categoria, meta_ventas
FROM analitica.metas_mensuales
ORDER BY anio, mes, categoria;

### ```UPDATE``` y ```DELETE```.

**Siempre** acompaña ```UPDATE``` y ```DELETE``` de un ```WHERE```; sin él, se modifican o eliminan todas las filas de la tabla.

In [ ]:
%%sql
UPDATE analitica.metas_mensuales
SET meta_ventas = meta_ventas * 1.05
WHERE anio = 2026 AND categoria = 'Jeans';

In [ ]:
%%sql
DELETE FROM analitica.metas_mensuales
WHERE anio = 2026 AND mes = 2;

> **Nota de dialecto:** *BigQuery* exige que todo ```UPDATE``` y ```DELETE``` tenga ```WHERE```; para afectar todas las filas se escribe ```WHERE TRUE```. Además, su *sandbox* no permite sentencias *DML*.

### ```MERGE```: el patrón de carga incremental.

En ingeniería de datos es común recibir un lote de cambios (*staging*) que contiene registros nuevos y registros modificados. ```MERGE``` (estándar desde *SQL:2003*) **actualiza los existentes e inserta los nuevos en una sola operación** (*upsert*).

* Tabla destino: clientes de Brasil.

In [ ]:
%%sql
CREATE TABLE analitica.clientes AS
SELECT id, first_name, last_name, country, state, created_at
FROM users
WHERE country = 'Brasil';

* Lote de cambios: los clientes de Brasil cuyo ```id``` es múltiplo de 100, con el estado modificado, más los clientes de Japón con ```id``` múltiplo de 100, que no existen en el destino.

In [ ]:
%%sql
CREATE TABLE analitica.clientes_cambios AS
SELECT id, first_name, last_name, country, 'Actualizado' AS state, created_at
FROM users
WHERE country = 'Brasil' AND MOD(id, 100) = 0
UNION ALL
SELECT id, first_name, last_name, country, state, created_at
FROM users
WHERE country = 'Japan' AND MOD(id, 100) = 0;

* Aplicación de los cambios.

In [ ]:
%%sql
MERGE INTO analitica.clientes AS destino
USING analitica.clientes_cambios AS cambios
  ON destino.id = cambios.id
WHEN MATCHED THEN
  UPDATE SET state = cambios.state
WHEN NOT MATCHED THEN
  INSERT (id, first_name, last_name, country, state, created_at)
  VALUES (cambios.id, cambios.first_name, cambios.last_name, cambios.country, cambios.state, cambios.created_at);

In [ ]:
%%sql
SELECT country, state, COUNT(*) AS clientes
FROM analitica.clientes
WHERE state = 'Actualizado' OR country = 'Japan'
GROUP BY country, state
ORDER BY country, state;

**Nota:** ```MERGE``` falla si una fila del destino coincide con más de una fila del lote de cambios. Por eso, antes de un ```MERGE```, el lote se **deduplica** con el patrón ```ROW_NUMBER()``` visto en la lección de funciones de ventana.

> **Nota de dialecto:** *DuckDB* admite ```MERGE``` desde la versión 1.4. *SQLite* y *MySQL* no lo tienen; usan ```INSERT ... ON CONFLICT``` e ```INSERT ... ON DUPLICATE KEY UPDATE```, respectivamente.

## Eliminación de objetos.

In [ ]:
%%sql
DROP VIEW analitica.v_clientes_por_pais;
DROP TABLE analitica.clientes_cambios;

> **Nota de dialecto:** ```DROP TABLE IF EXISTS``` evita el error cuando la tabla no existe; no es estándar, pero lo admiten casi todos los motores. Para eliminar un esquema con todo su contenido: ```DROP SCHEMA analitica CASCADE;```.

## Ejercicios.

1. Crea la tabla ```analitica.resumen_clientes``` con una fila por cliente que haya comprado: ```user_id```, número de órdenes, fecha de la primera y de la última orden.
2. Crea la vista ```analitica.v_productos_sin_venta``` con los productos que nunca se han vendido. ¿Cuántas filas tiene?
3. Inserta en ```analitica.metas_mensuales``` una meta para enero de 2026 de cada categoría, igual a sus ventas de enero de 2025 más un 15 %. Cuida no duplicar las metas que ya existen (pista: ```NOT EXISTS```).
4. Escribe un ```MERGE``` que actualice ```analitica.ventas_mensuales``` a partir de un lote con las ventas de diciembre de 2025 recalculadas incluyendo también los artículos con estado ```Shipped```: actualiza las filas existentes e inserta las nuevas.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2026.</p>